In [ ]:
# - for a leaf tensor, grad_fn == None (always, even if requries_grad == Tue). Actually a leaf tensor is defined as a tensor with no grad_fn.
# - for an intermediate tensor, which is the result of an operation AND requires_grad = True, grad_fn exists (points to tthe operation that created it)
# - if x.requires_grad == False, then grad_fn == None (always)

# This is the precise definition used in PyTorch:
# Leaf = grad_fn is None (requires_grad may or may not be True)
# Non-leaf = grad_fn is not None (has a history) (So requires_grad == True)

grad_fn               requires_grad                Possible?

None                  True                         Yes (leaf)
None                  False                        Yes (leaf)
Not None              True                         Yes (non-leaf)
Not None              False                        No


# If a tensor has a grad_fn (meaning it was produced by an operation), 
# then at least one of its inputs required gradients, which forces requires_grad = True on the output.

# requires_grad = False  ⇒  grad_fn is None

x.grad_fn is not None, x.requries_grad is True.
If x is detatched, x.grad_fn is None and x.requires_grad is False.
So, x is now a leaf.
So, we can change x.requires_grad.

When you call .requires_grad_(False) on a non-leaf tensor, PyTorch automatically detaches it.

Object `Possible` not found.


In [ ]:
import torch

In [ ]:
x = torch.tensor([2.0], requires_grad=True)
y = x ** 2

print(y.requires_grad)  # True
print(y.grad_fn)        # <PowBackward0 object>

y.backward()
print(x.grad)  # None (gradient didn't flow through)

In [ ]:

x = torch.tensor([2.0], requires_grad=True)
y = x ** 2
z = y.detach()  # No gradient connection to x ---- no requires_grad, no grad_fn
w = z ** 2

print(y.requires_grad)  # True
print(z.requires_grad)  # False
print(z.grad_fn)        # None

# w.grad_fn is None because z.requires_grad = False, and when all inputs to an operation have requires_grad = False, 
# the output also has requires_grad = False and no grad_fn is created.  --- I agree!
print(w.grad_fn)        # None

# It cannot backward because w.grad_fn == None.
w.backward()
print(x.grad)  # None (gradient didn't flow through)

In [ ]:
x = torch.tensor([2.0], requires_grad=True)
y = x ** 2
z = y.detach().requires_grad_(True)  # New leaf tensor with gradient tracking
w = z ** 2

print(z.grad)  # None
print(x.grad)  # None
print(w.grad_fn)        # <PowBackward0 object> is created, becasue z.rquires_grad == True and we need w.grad_fn, which stores d_w / d_z, which is used to find z.grad.

# w.grad_fn exists ⇔ w.requires_grad = True ⇔ at least one input requires grad

# It can backward because we have w.grad_fn. But if z.requires_grad is False, it will not backward.
w.backward()

print(z.grad)  # tensor([8.]) (∂w/∂z = 2*z = 2*4 = 8)
print(x.grad)  # None (graph broken by detach, no gradient flows to x)
print(w.grad_fn)        # None


None
None
tensor([8.])
None


In [37]:
x = torch.tensor([2.0], requires_grad=True)
y = x ** 2
y.detach_()  # In-place detach (modifies y)
w = y ** 2

w.backward()    # element 0 of tensors does not require grad and does not have a grad_fn
print(x.grad)  # None

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

In [ ]:
x = torch.tensor([2.0], requires_grad=True)

with torch.no_grad():
    y = x ** 2  # y doesn't track gradients. torch doesn't build y.grad_fn and y.requires_grad == False
    z = y ** 2  # z doesn't track gradients

print(y.requires_grad)  # False
print(z.requires_grad)  # False

# Can't call backward on z because no graph

False
False


In [ ]:
x = torch.tensor([2.0], requires_grad=True)
y = x ** 2

# Hook that zeros out gradients
def zero_grad_hook(grad):
    # return torch.zeros_like(grad)
    return 2 * grad

y.register_hook(zero_grad_hook)

z = y ** 2
z.backward()
print(y.requires_grad)
print(y.grad) # None. The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. The .grad is not retained after use during backward, even if y.reequires_grad is True.
print(x.grad)  # tensor([0.]) (gradient zeroed out)

True
None
tensor([64.])


C:\Users\Mike\AppData\Local\Temp\ipykernel_17496\4031379953.py:14: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more informations. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\build\aten\src\ATen/core/TensorBody.h:494.)
  print(y.grad) # None. The .grad attribute of a Tensor that is not a leaf Tensor is being accessed.


In [ ]:
x = torch.tensor([2.0], requires_grad=True)
y = x ** 2

print(y.grad_fn)
# you can only change requires_grad flags of leaf variables. 
# If you want to use a computed variable in a subgraph that doesn't require differentiation use var_no_grad = var.detach()
y.requires_grad = False #(False)  # You can only change requires_gradd of leaf variables. Use .detach()
print(y.grad_fn) 
z = y ** 2

# z.backward()  # Error! Can't backward through y because it doesn't track gradients

RuntimeError: you can only change requires_grad flags of leaf variables. If you want to use a computed variable in a subgraph that doesn't require differentiation use var_no_grad = var.detach().

In [35]:
x = torch.tensor([2.0], requires_grad=True)
y = x ** 2  # Power operation
z = y + 3   # Addition operation
loss = z ** 2  # Another power operation

loss.backward()

# Inspect loss.grad_fn
print("=== loss.grad_fn ===")
print(f"Type: {type(loss.grad_fn)}")
print(f"Name: {loss.grad_fn.name()}")
print(f"next_functions: {loss.grad_fn.next_functions}")

print(f"Type: {type(z.grad_fn)}")
print(f"Name: {z.grad_fn.name()}")
print(f"next_functions: {z.grad_fn.next_functions}")

=== loss.grad_fn ===
Type: <class 'PowBackward0'>
Name: PowBackward0
next_functions: ((<AddBackward0 object at 0x0000020AFBBDB190>, 0),)
Type: <class 'AddBackward0'>
Name: AddBackward0
next_functions: ((<PowBackward0 object at 0x0000020AFBBD8820>, 0), (None, 0))
